In [1]:
import joblib
import pandas as pd
from sklearn import set_config
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from pathlib import Path
import datetime as dt
import warnings
import time
import pytz
import json
import os

In [2]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-08-08-05_41_27_PM'

In [3]:
experiment_config = {
    "experiment": {
        "id": f"{dt_str}_xgb",
        "model": "xgb",
        "type": "baseline",
        "description": "basic xgboost baseline",
    },

    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 0
    },

    "params": {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 1500,
        "learning_rate": 0.1,
        "max_depth": 5,
        "tree_method": "hist",
        "enable_categorical": True,
        "early_stopping_rounds": 10
    }
}

In [4]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)
experiment_path

PosixPath('/kaggle/working/experiments/2026-08-08-05_41_27_PM_xgb')

In [5]:
with open(experiment_path / "config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

In [6]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [7]:
X = pd.read_csv(f"{data_path}/processed/train_features.csv")
X_test = pd.read_csv(f"{data_path}/processed/test_features.csv")
y = pd.read_csv(f"{data_path}/processed/train_labels.csv")

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 23 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               662440 non-null  float64
 1   daily_screen_time_hours           595515 non-null  float64
 2   social_media_hours                557374 non-null  float64
 3   gaming_hours                      564548 non-null  float64
 4   work_study_hours                  639851 non-null  float64
 5   sleep_hours                       646889 non-null  float64
 6   notifications_per_day             623785 non-null  float64
 7   app_opens_per_day                 610659 non-null  float64
 8   weekend_screen_time               579306 non-null  float64
 9   gender                            662335 non-null  object 
 10  stress_level                      636221 non-null  float64
 11  academic_work_impact              647145 non-null  f

In [8]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns

for frame in [X, X_test]:
    for col in cat_cols:
        frame[col] = frame[col].astype('category')

In [9]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 23 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   age                               662440 non-null  float64 
 1   daily_screen_time_hours           595515 non-null  float64 
 2   social_media_hours                557374 non-null  float64 
 3   gaming_hours                      564548 non-null  float64 
 4   work_study_hours                  639851 non-null  float64 
 5   sleep_hours                       646889 non-null  float64 
 6   notifications_per_day             623785 non-null  float64 
 7   app_opens_per_day                 610659 non-null  float64 
 8   weekend_screen_time               579306 non-null  float64 
 9   gender                            662335 non-null  category
 10  stress_level                      636221 non-null  float64 
 11  academic_work_impact              64714

In [10]:
kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

y_cv = pd.Series(index=y.index, dtype=float, name='predicted_proba')

fold_scores = []

start_time = time.time()

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    model = XGBClassifier(**experiment_config["params"])
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])

    y_pred = model.predict_proba(X_valid)[:, 1]

    y_cv.iloc[valid_index] = y_pred

    fold_auc_score = roc_auc_score(y_valid, y_pred)
    fold_scores.append(fold_auc_score)

elapsed = time.time() - start_time

y_pred_df = y_cv.to_frame()
y_pred_df.to_csv(experiment_path / "oof.csv", index=False)

[0]	validation_0-auc:0.92211
[1]	validation_0-auc:0.92424
[2]	validation_0-auc:0.92732
[3]	validation_0-auc:0.92937
[4]	validation_0-auc:0.93012
[5]	validation_0-auc:0.93026
[6]	validation_0-auc:0.93022
[7]	validation_0-auc:0.93029
[8]	validation_0-auc:0.93045
[9]	validation_0-auc:0.93044
[10]	validation_0-auc:0.93126
[11]	validation_0-auc:0.93119
[12]	validation_0-auc:0.93120
[13]	validation_0-auc:0.93116
[14]	validation_0-auc:0.93116
[15]	validation_0-auc:0.93159
[16]	validation_0-auc:0.93193
[17]	validation_0-auc:0.93206
[18]	validation_0-auc:0.93261
[19]	validation_0-auc:0.93272
[20]	validation_0-auc:0.93316
[21]	validation_0-auc:0.93330
[22]	validation_0-auc:0.93359
[23]	validation_0-auc:0.93387
[24]	validation_0-auc:0.93410
[25]	validation_0-auc:0.93428
[26]	validation_0-auc:0.93452
[27]	validation_0-auc:0.93475
[28]	validation_0-auc:0.93499
[29]	validation_0-auc:0.93506
[30]	validation_0-auc:0.93532
[31]	validation_0-auc:0.93563
[32]	validation_0-auc:0.93577
[33]	validation_0-au

In [11]:
valid_auc_score = roc_auc_score(y, y_cv)

print("Validation AUC:", valid_auc_score)

Validation AUC: 0.9639873699375698


In [12]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "random_state": 0,
        "fold_scores": fold_scores,
        "mean": sum(fold_scores) / len(fold_scores),
        "std": float(pd.Series(fold_scores).std(ddof=1))
    },
    "primary_metric": {
        "name": "auc",
        "value": valid_auc_score
    },
    "training": {
        "duration_seconds": elapsed
    }
}

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

In [13]:
experiment_config["params"]["early_stopping_rounds"] = None
model = XGBClassifier(**experiment_config["params"])
model.fit(X, y)

joblib.dump(model, experiment_path / f"{experiment_config["experiment"]["model"]}.pkl")

['/kaggle/working/experiments/2026-08-08-05_41_27_PM_xgb/xgb.pkl']

In [14]:
y_pred = model.predict_proba(X_test)
ss[target_column] = y_pred

ss.to_csv(experiment_path / "submission.csv", index=False)
ss

,id,addicted_label
0,691369,2.496839e-04
1,691370,9.280080e-02
2,691371,5.178285e-02
3,691372,1.009810e-02
4,691373,6.973743e-04
...,...,...
296297,987666,1.192093e-07
296298,987667,1.155071e-01
296299,987668,7.906196e-01
296300,987669,2.374858e-01
